# Lesson 11a: Language Model Pretraining — Theory

10a assembled the Transformer block: multi-head self-attention, positional
encodings, residual/pre-norm structure, and causal masking. This lesson asks
what those pieces are trained to do. **Pretraining** a language model means
training it to predict the next token of a sequence, over and over, on raw
text with no labels beyond the text itself — the self-supervision that lets
one generic objective scale to any amount of unlabelled text. This notebook
derives that objective precisely (the **autoregressive** factorisation and
its **cross-entropy** loss), defines the standard way progress on it is
reported (**perplexity**), gives an intuitive, empirically-grounded account
of how that loss responds to **model size, data, and compute**, and closes
by assembling 10a's Transformer block, causal mask, and positional
encodings into a complete **decoder-only mini-GPT**, whose from-scratch
NumPy forward pass and loss are checked exactly against an equivalent
PyTorch computation. Data: **Tiny Shakespeare**, tokenised at the character
level — 11b trains this same architecture for real and compares it against
a pretrained GPT-2.

By the end of this notebook you will have:
- derived the **autoregressive objective** and shown its negative
  log-likelihood is exactly the **cross-entropy loss** already used for
  classification (2a, 8a),
- defined **perplexity** and confirmed, exactly, that an untrained
  (uniform-guessing) model's perplexity equals the vocabulary size,
- trained (very small, very fast) neural language models of varying width
  and varying data budget to show, empirically, how loss falls with
  **parameters** and with **data**, and derived the standard **compute**
  approximation that ties both to a single budget, and
- assembled a full **decoder-only mini-GPT** — embeddings, positional
  encoding, causal Transformer blocks, output head — from scratch in
  NumPy, and verified its forward pass and loss against PyTorch to
  floating-point precision.

## Introduction

Every notebook so far has trained on an explicit label: 2a/2b's MNIST
digit, 3a's CIFAR-10 class, 8a's neighbouring word. A language model's
training signal is different in kind, not just in content: the label for
predicting the token at position $t$ is simply the token that *actually
appears* at position $t$ in the raw text — no separate annotation is ever
required. This is why pretraining scales the way it does: any text at all,
scraped or typed or transcribed, supplies its own supervision for free, at a
scale hand-labelled datasets cannot approach. Everything else in this
notebook works out the consequences of that one idea: the **objective** the
model is actually optimising ("The Autoregressive Objective"), the standard
unit progress on it is measured in ("Cross-Entropy and Perplexity"), an
empirical account of what happens as the model, the data, and the compute
spent on both are scaled up ("Scaling Intuition"), and, finally, the
architecture — 10a's Transformer block, arranged causally — that turns the
objective into a trainable model ("Mini-GPT from Scratch").

## Setup

In [ ]:
# Fixed seeds: every stochastic step (weight init, minibatch sampling,
# data subsampling) is reproducible.
import pathlib
import urllib.request

import numpy as np
import torch
import torch.nn.functional as F

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)


def softmax(x, axis=-1):
    shifted = x - x.max(axis=axis, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=axis, keepdims=True)

### Tiny Shakespeare

The corpus is the complete works of Shakespeare concatenated into one
~1.1M-character file — small enough to download and hold in memory in a
fraction of a second, large enough to train a real (if tiny) language
model on. Tokenising at the **character** level keeps the vocabulary to a
few dozen symbols and sidesteps the subword-tokeniser question 8a/8b
already covered in depth; the pretraining objective and the mini-GPT
architecture below are entirely independent of *which* tokeniser produces
the integer sequence they are trained on.

In [ ]:
DATA_DIR = pathlib.Path("data")
DATA_DIR.mkdir(exist_ok=True)
CORPUS_PATH = DATA_DIR / "tinyshakespeare.txt"
CORPUS_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not CORPUS_PATH.exists():
    urllib.request.urlretrieve(CORPUS_URL, CORPUS_PATH)
full_text = CORPUS_PATH.read_text(encoding="utf-8")

chars = sorted(set(full_text))
VOCAB_SIZE = len(chars)
char_to_id = {c: i for i, c in enumerate(chars)}
id_to_char = {i: c for i, c in enumerate(chars)}


def encode(s):
    return np.array([char_to_id[c] for c in s], dtype=np.int64)


def decode(ids):
    return "".join(id_to_char[int(i)] for i in ids)


N_CHARS = 20000
text_slice = full_text[:N_CHARS]
encoded = encode(text_slice)

print(f"corpus: {len(full_text)} characters total, vocabulary: {VOCAB_SIZE} unique characters")
print(f"using the first {N_CHARS} characters for the demonstrations in this notebook")
print(f"sample: {full_text[:80]!r}")

## The Autoregressive Objective

The chain rule of probability is an exact identity for *any* joint
distribution over a sequence $x_1,\dots,x_T$ — no modelling assumption is
involved yet:

$$p(x_1,\dots,x_T) = \prod_{t=1}^{T} p(x_t \mid x_1,\dots,x_{t-1}) = \prod_{t=1}^{T} p(x_t \mid x_{<t}).$$

An **autoregressive** language model chooses to parameterise the
right-hand side's conditionals directly, with one model
$p_\theta(x_t\mid x_{<t})$ **shared across every position and every
training sequence** — the same weight-sharing idea 7a's recurrence and
5a's convolution kernel already use, applied here to the modelling
assumption itself rather than to a specific layer. Given an observed
training sequence, maximum likelihood chooses $\theta$ to make that
sequence as probable as possible under the model, which (taking a negative
log, then averaging per token so sequence length does not dominate the
objective) is equivalent to minimising:

$$\mathcal{L}(\theta) = -\frac{1}{T}\sum_{t=1}^{T} \log p_\theta(x_t \mid x_{<t}).$$

At a single position $t$, $p_\theta(\cdot \mid x_{<t})$ is a categorical
distribution over the vocabulary — a softmax over logits, exactly as in
every classifier this series has built — and $x_t$ is the one true class.
So $-\log p_\theta(x_t\mid x_{<t})$ **is** the cross-entropy loss from
2a/8a, applied independently at every position, with 10a's causal mask
being exactly the mechanism that supplies $x_{<t}$ (and nothing more) to
the model computing $p_\theta(\cdot\mid x_{<t})$. "Pretraining objective"
is therefore not a new loss function — it is the familiar classification
cross-entropy, applied $T$ times per sequence, at a scale unlabelled text
can support that labelled classification data cannot.

In [ ]:
# The autoregressive negative log-likelihood and the one-hot cross-entropy
# are the same formula written two ways -- confirm they give the identical
# number for an arbitrary categorical prediction, not just in the abstract.
V_TOY, T_TOY = 6, 5
rng = np.random.default_rng(SEED)
logits_toy = rng.normal(size=(T_TOY, V_TOY))
targets_toy = rng.integers(0, V_TOY, size=T_TOY)
probs_toy = softmax(logits_toy)

nll = -np.mean(np.log(probs_toy[np.arange(T_TOY), targets_toy]))

one_hot_targets = np.eye(V_TOY)[targets_toy]
cross_entropy = -np.mean(np.sum(one_hot_targets * np.log(probs_toy), axis=1))

print(f"negative log-likelihood (autoregressive form): {nll:.10f}")
print(f"cross-entropy (one-hot classification form):    {cross_entropy:.10f}")
assert np.isclose(nll, cross_entropy)

## Cross-Entropy and Perplexity

Reporting a loss in nats is not intuitive on its own — few people have
an instinct for "2.1 nats per token." **Perplexity** re-expresses the same
number in units that are:

$$\text{PPL} = \exp(\mathcal{L}) = \exp\!\left(-\frac{1}{T}\sum_{t=1}^{T} \log p_\theta(x_t\mid x_{<t})\right).$$

Perplexity is the geometric mean of $1/p_\theta(x_t\mid x_{<t})$ across
every position: "the model is, on average, as uncertain as if it were
choosing uniformly among PPL equally likely options at every step." That
gives an immediate, concrete calibration point: a model that predicts the
**uniform** distribution over the vocabulary at every position, regardless
of context, has $p_\theta(x_t\mid x_{<t}) = 1/V$ for every $t$, so

$$\mathcal{L}_{\text{uniform}} = -\log(1/V) = \log V, \qquad \text{PPL}_{\text{uniform}} = V.$$

An untrained model's weights are small and close to symmetric, so its
predictions start close to uniform — any trained model that has not beaten
"perplexity $\approx$ vocabulary size" has not learned anything about the
data at all.

In [ ]:
# A prediction that is exactly uniform over the vocabulary gives a loss of
# exactly log(V) and a perplexity of exactly V -- for ANY targets, since a
# uniform distribution assigns every class the same probability 1/V.
N_PROBE = 200
rng2 = np.random.default_rng(SEED)
logits_uniform = np.zeros((N_PROBE, VOCAB_SIZE))  # all-zero logits -> softmax is exactly uniform
targets_probe = rng2.integers(0, VOCAB_SIZE, size=N_PROBE)

probs_uniform = softmax(logits_uniform)
loss_uniform = -np.mean(np.log(probs_uniform[np.arange(N_PROBE), targets_probe]))
ppl_uniform = np.exp(loss_uniform)

print(f"vocabulary size:                       {VOCAB_SIZE}")
print(f"loss under a uniform prediction:       {loss_uniform:.6f} nats  (ln({VOCAB_SIZE}) = {np.log(VOCAB_SIZE):.6f})")
print(f"perplexity under a uniform prediction: {ppl_uniform:.3f}        (= vocabulary size)")
assert np.isclose(loss_uniform, np.log(VOCAB_SIZE))
assert np.isclose(ppl_uniform, VOCAB_SIZE)

## Scaling Intuition

Kaplan et al. (2020) and Hoffmann et al. (2022, "Chinchilla") measured
that a Transformer language model's test loss follows a smooth power law
in each of parameters $N$, dataset size $D$, and training compute $C$, when
the other two are not the bottleneck:

$$L(N) \approx \left(\frac{N_c}{N}\right)^{\alpha_N}, \qquad L(D) \approx \left(\frac{D_c}{D}\right)^{\alpha_D},$$

with measured exponents around $\alpha_N \approx 0.076$, $\alpha_D \approx
0.095$ — **these are empirical fits, not derivations**, unlike every other
equation in this notebook. Training compute for a Transformer (one forward
plus one backward pass over every training token, each parameter touched
roughly twice forward and four times backward) is well approximated by

$$C \approx 6\,N\,D$$

(tokens $\times$ parameters $\times$ a small constant; the constant 6 is
itself an approximation, not exact). This ties the three axes together:
for a fixed compute budget $C$, there is a tradeoff between a bigger $N$
and more $D$, and Chinchilla's central finding was that prior large models
were substantially *undertrained* for their size — compute-optimal
training scales $N$ and $D$ at roughly the same rate ($N \propto D \propto
\sqrt{C}$), a specific, falsifiable claim that changed how large models
have been trained since.

This notebook cannot reproduce those laws literally — they were measured
across models spanning many orders of magnitude of $N$ and $D$, trained far
beyond a CPU notebook's ten-minute budget. What it can do is show the same
*qualitative* shape at toy scale: a small feed-forward next-character
predictor (a fixed window of the previous $k$ characters, not the
attention mechanism built below) trained at several widths and several
data budgets, to confirm loss falls with both.

In [ ]:
class TinyCharLM(torch.nn.Module):
    """Predict the next character from the previous CONTEXT_K characters:
    embed each, concatenate, one hidden layer. Deliberately simpler than the
    Transformer built below -- only the width/data/compute trend matters
    here, not the architecture."""

    def __init__(self, vocab_size, context_k, hidden_dim):
        super().__init__()
        self.embed = torch.nn.Embedding(vocab_size, hidden_dim)
        self.fc1 = torch.nn.Linear(hidden_dim * context_k, hidden_dim)
        self.fc2 = torch.nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):  # x: (batch, context_k) long
        e = self.embed(x).reshape(x.shape[0], -1)
        return self.fc2(F.relu(self.fc1(e)))


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def make_context_dataset(ids, context_k):
    xs = np.stack([ids[i:i + context_k] for i in range(len(ids) - context_k)])
    ys = ids[context_k:]
    return xs, ys


def train_tiny_char_lm(hidden_dim, train_ids, context_k, steps, batch_size, eval_ids, seed=SEED):
    """Trains on train_ids, then reports loss on a FIXED held-out slice
    (eval_ids) rather than the last training batch. Measuring training-batch
    loss under a fixed step budget confounds "more data" with "less
    memorisation per example" -- a smaller train_ids gets revisited more
    often in the same number of steps and its training loss falls mostly
    from memorising it, not from generalising. Held-out loss is what the
    scaling-law literature actually reports."""
    torch.manual_seed(seed)
    X, Y = make_context_dataset(train_ids, context_k)
    X_t = torch.tensor(X, dtype=torch.long)
    Y_t = torch.tensor(Y, dtype=torch.long)
    model = TinyCharLM(VOCAB_SIZE, context_k, hidden_dim)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    g = torch.Generator().manual_seed(seed)
    n = len(X_t)
    for _ in range(steps):
        idx = torch.randint(0, n, (batch_size,), generator=g)
        logits = model(X_t[idx])
        loss = F.cross_entropy(logits, Y_t[idx])
        opt.zero_grad()
        loss.backward()
        opt.step()

    X_val, Y_val = make_context_dataset(eval_ids, context_k)
    with torch.no_grad():
        val_logits = model(torch.tensor(X_val, dtype=torch.long))
        val_loss = F.cross_entropy(val_logits, torch.tensor(Y_val, dtype=torch.long)).item()
    return model, val_loss


CONTEXT_K, STEPS, BATCH_SIZE = 8, 200, 64
WIDTHS = [8, 16, 32, 64, 128]
DATA_SIZES = [500, 1000, 2000, 4000, 8000]
FIXED_DATA_FOR_WIDTH_SWEEP = 10000
FIXED_WIDTH_FOR_DATA_SWEEP = 32
# A fixed held-out block, disjoint from every training slice above
# (the largest of which is encoded[:10000]) -- the same yardstick for
# every run in both sweeps.
EVAL_IDS = encoded[10000:20000]

width_results = []
for hidden_dim in WIDTHS:
    model, loss = train_tiny_char_lm(
        hidden_dim, encoded[:FIXED_DATA_FOR_WIDTH_SWEEP], CONTEXT_K, STEPS, BATCH_SIZE, EVAL_IDS
    )
    width_results.append((count_params(model), loss))

data_results = []
for d in DATA_SIZES:
    model, loss = train_tiny_char_lm(FIXED_WIDTH_FOR_DATA_SWEEP, encoded[:d], CONTEXT_K, STEPS, BATCH_SIZE, EVAL_IDS)
    data_results.append((d, loss))

print("width sweep (fixed data, fixed steps) -- held-out loss:")
for n_params, loss in width_results:
    print(f"  N={n_params:6d} params  held-out loss={loss:.4f}")
print("data sweep (fixed width, fixed steps) -- held-out loss:")
for d, loss in data_results:
    print(f"  D={d:6d} chars    held-out loss={loss:.4f}")


In [ ]:
fig, (ax_n, ax_d) = plt.subplots(1, 2, figsize=(11, 4.5))

ns, n_losses = zip(*width_results)
ax_n.plot(ns, n_losses, marker="o")
ax_n.set_xscale("log")
ax_n.set_xlabel("parameters $N$ (log scale)")
ax_n.set_ylabel("final training loss (nats)")
ax_n.set_title(f"Loss vs. parameters (fixed D={FIXED_DATA_FOR_WIDTH_SWEEP} chars)")
ax_n.grid(alpha=0.3)

ds, d_losses = zip(*data_results)
ax_d.plot(ds, d_losses, marker="o", color="tab:orange")
ax_d.set_xscale("log")
ax_d.set_xlabel("training characters $D$ (log scale)")
ax_d.set_ylabel("final training loss (nats)")
ax_d.set_title(f"Loss vs. data (fixed N, width={FIXED_WIDTH_FOR_DATA_SWEEP})")
ax_d.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Both curves fall as their respective axis grows — more parameters at
fixed data, and more data at fixed parameters, both reduce the loss a
tiny character-level model reaches in the same fixed number of gradient
steps, the same qualitative shape Kaplan/Chinchilla report at vastly
larger scale (the toy exponents here are not meaningful — five points
spanning one order of magnitude cannot pin down a power-law exponent; only
the direction and smoothness of the trend transfer). The table below makes
the $C\approx 6ND$ compute link concrete for the width sweep, where every
run processes the same number of tokens ($D_{\text{tokens}} =
\text{steps}\times\text{batch size}$, via repeated sampling) so compute
scales with $N$ alone:

In [ ]:
tokens_processed = STEPS * BATCH_SIZE  # every width-sweep run samples this many tokens total
print(f"{'N (params)':>12} {'D (tokens processed)':>22} {'C = 6ND (FLOPs)':>18} {'held-out loss':>12}")
for n_params, loss in width_results:
    compute = 6 * n_params * tokens_processed
    print(f"{n_params:>12,} {tokens_processed:>22,} {compute:>18,} {loss:>12.4f}")

Larger $N$ costs proportionally more compute for the same number of
tokens processed, and (over this range) buys a lower loss in exchange —
exactly the tradeoff a fixed real-world compute budget forces a choice
between. A compute-optimal choice would grow $D$ alongside $N$, per
Chinchilla, rather than holding it fixed as this deliberately isolated
sweep does.

## Mini-GPT from Scratch

Now assemble 10a's pieces into a complete model: token embeddings,
sinusoidal positional encoding, $L$ stacked causal, pre-norm Transformer
blocks, a final layer norm, and a linear head projecting to vocabulary
logits — a decoder-only Transformer, commonly called a (mini-)GPT. Trained
(in 11b) with exactly the cross-entropy loss derived above, under the
causal mask that supplies each position with $x_{<t}$ and nothing more,
this is the architecture behind every autoregressive language model in
this lesson's lineage. This section builds its forward pass and loss from
scratch in NumPy, for a single batch of real Tiny Shakespeare sequences,
and checks the result against an identical computation built from the
*same* weights in PyTorch — the verification discipline 9a and 10a already
applied to attention and to one Transformer block, now applied to the
complete model.

In [ ]:
def sinusoidal_positional_encoding(max_len, d_model):
    pos = np.arange(max_len)[:, None]
    i = np.arange(d_model)[None, :]
    angle_rates = 1.0 / (10000 ** (2 * (i // 2) / d_model))
    angles = pos * angle_rates
    pe = np.zeros((max_len, d_model))
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])
    return pe


def causal_mask(T):
    return np.triu(np.ones((T, T)), k=1).astype(bool)  # True where j > i (future positions)


def layer_norm_np(x, gamma, beta, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return gamma * (x - mu) / np.sqrt(var + eps) + beta


def multi_head_causal_attention_np(X, Wq, Wk, Wv, Wo, num_heads, mask):
    T, d_model = X.shape
    d_k = d_model // num_heads
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    Qh = Q.reshape(T, num_heads, d_k).transpose(1, 0, 2)
    Kh = K.reshape(T, num_heads, d_k).transpose(1, 0, 2)
    Vh = V.reshape(T, num_heads, d_k).transpose(1, 0, 2)
    outputs = []
    for h in range(num_heads):
        scores = (Qh[h] @ Kh[h].T) / np.sqrt(d_k)
        scores = np.where(mask, -np.inf, scores)
        outputs.append(softmax(scores) @ Vh[h])
    return np.concatenate(outputs, axis=-1) @ Wo


def ffn_np(x, W1, b1, W2, b2):
    return np.maximum(0.0, x @ W1 + b1) @ W2 + b2


def transformer_block_forward_np(x, blk, mask, num_heads):
    normed = layer_norm_np(x, blk["ln1_g"], blk["ln1_b"])
    x = x + multi_head_causal_attention_np(normed, blk["Wq"], blk["Wk"], blk["Wv"], blk["Wo"], num_heads, mask)
    normed2 = layer_norm_np(x, blk["ln2_g"], blk["ln2_b"])
    return x + ffn_np(normed2, blk["W1"], blk["b1"], blk["W2"], blk["b2"])


def mini_gpt_forward_np(tokens, params, mask, num_heads):
    T = tokens.shape[0]
    x = params["E"][tokens] + params["pe"][:T]
    for blk in params["blocks"]:
        x = transformer_block_forward_np(x, blk, mask, num_heads)
    x = layer_norm_np(x, params["lnf_g"], params["lnf_b"])
    return x @ params["W_head"] + params["b_head"]


def cross_entropy_loss_np(logits, targets):
    probs = softmax(logits)
    T = logits.shape[0]
    return -np.mean(np.log(np.clip(probs[np.arange(T), targets], 1e-300, 1.0)))


def init_mini_gpt_params(rng, vocab_size, d_model, num_heads, d_ff, num_layers, max_len, scale=0.02):
    blocks = []
    for _ in range(num_layers):
        blocks.append(dict(
            ln1_g=np.ones(d_model), ln1_b=np.zeros(d_model),
            Wq=rng.normal(scale=scale, size=(d_model, d_model)),
            Wk=rng.normal(scale=scale, size=(d_model, d_model)),
            Wv=rng.normal(scale=scale, size=(d_model, d_model)),
            Wo=rng.normal(scale=scale, size=(d_model, d_model)),
            ln2_g=np.ones(d_model), ln2_b=np.zeros(d_model),
            W1=rng.normal(scale=scale, size=(d_model, d_ff)), b1=np.zeros(d_ff),
            W2=rng.normal(scale=scale, size=(d_ff, d_model)), b2=np.zeros(d_model),
        ))
    return dict(
        E=rng.normal(scale=scale, size=(vocab_size, d_model)),
        pe=sinusoidal_positional_encoding(max_len, d_model),
        blocks=blocks,
        lnf_g=np.ones(d_model), lnf_b=np.zeros(d_model),
        W_head=rng.normal(scale=scale, size=(d_model, vocab_size)),
        b_head=np.zeros(vocab_size),
    )


D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS, BLOCK_T, BATCH_B = 32, 4, 64, 2, 16, 4

rng_gpt = np.random.default_rng(SEED)
params_np = init_mini_gpt_params(rng_gpt, VOCAB_SIZE, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS, max_len=BLOCK_T)
mask = causal_mask(BLOCK_T)

# A real batch of Tiny Shakespeare windows: BATCH_B non-overlapping chunks of
# BLOCK_T+1 characters (input = first BLOCK_T, target = the next BLOCK_T,
# i.e. next-character prediction at every position).
tokens_batch = np.stack([encoded[b * (BLOCK_T + 1): b * (BLOCK_T + 1) + BLOCK_T] for b in range(BATCH_B)])
targets_batch = np.stack([encoded[b * (BLOCK_T + 1) + 1: b * (BLOCK_T + 1) + 1 + BLOCK_T] for b in range(BATCH_B)])

logits_np = np.stack([mini_gpt_forward_np(tokens_batch[b], params_np, mask, NUM_HEADS) for b in range(BATCH_B)])
loss_np = np.mean([cross_entropy_loss_np(logits_np[b], targets_batch[b]) for b in range(BATCH_B)])

n_learned_params = sum(
    v.size for blk in params_np["blocks"] for v in blk.values()
) + params_np["E"].size + params_np["lnf_g"].size + params_np["lnf_b"].size +     params_np["W_head"].size + params_np["b_head"].size

print(f"mini-GPT: {NUM_LAYERS} layers, d_model={D_MODEL}, {NUM_HEADS} heads, {n_learned_params:,} learned parameters")
print(f"input batch:  {tokens_batch.shape} (batch, block_size)")
print(f"logits shape: {logits_np.shape} (batch, block_size, vocab)")
print(f"NumPy loss on this batch: {loss_np:.6f} nats  (perplexity {np.exp(loss_np):.2f})")

In [ ]:
def layer_norm_torch(x, gamma, beta, eps=1e-5):
    mu = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    return gamma * (x - mu) / torch.sqrt(var + eps) + beta


def multi_head_causal_attention_torch(X, Wq, Wk, Wv, Wo, num_heads, mask_bool):
    T, d_model = X.shape
    d_k = d_model // num_heads
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    Qh = Q.reshape(T, num_heads, d_k).permute(1, 0, 2)
    Kh = K.reshape(T, num_heads, d_k).permute(1, 0, 2)
    Vh = V.reshape(T, num_heads, d_k).permute(1, 0, 2)
    outputs = []
    for h in range(num_heads):
        scores = (Qh[h] @ Kh[h].T) / (d_k ** 0.5)
        scores = scores.masked_fill(mask_bool, float("-inf"))
        outputs.append(torch.softmax(scores, dim=-1) @ Vh[h])
    return torch.cat(outputs, dim=-1) @ Wo


def ffn_torch(x, W1, b1, W2, b2):
    return torch.relu(x @ W1 + b1) @ W2 + b2


def transformer_block_forward_torch(x, blk, mask_bool, num_heads):
    normed = layer_norm_torch(x, blk["ln1_g"], blk["ln1_b"])
    x = x + multi_head_causal_attention_torch(normed, blk["Wq"], blk["Wk"], blk["Wv"], blk["Wo"], num_heads, mask_bool)
    normed2 = layer_norm_torch(x, blk["ln2_g"], blk["ln2_b"])
    return x + ffn_torch(normed2, blk["W1"], blk["b1"], blk["W2"], blk["b2"])


def mini_gpt_forward_torch(tokens, params_t, mask_bool, num_heads):
    T = tokens.shape[0]
    x = params_t["E"][tokens] + params_t["pe"][:T]
    for blk in params_t["blocks"]:
        x = transformer_block_forward_torch(x, blk, mask_bool, num_heads)
    x = layer_norm_torch(x, params_t["lnf_g"], params_t["lnf_b"])
    return x @ params_t["W_head"] + params_t["b_head"]


def cross_entropy_loss_torch(logits, targets):
    log_probs = torch.log_softmax(logits, dim=-1)
    T = logits.shape[0]
    return -log_probs[torch.arange(T), targets].mean()


def to_torch(x):
    return torch.tensor(x)


params_t = dict(
    E=to_torch(params_np["E"]),
    pe=to_torch(params_np["pe"]),
    blocks=[{k: to_torch(v) for k, v in blk.items()} for blk in params_np["blocks"]],
    lnf_g=to_torch(params_np["lnf_g"]), lnf_b=to_torch(params_np["lnf_b"]),
    W_head=to_torch(params_np["W_head"]), b_head=to_torch(params_np["b_head"]),
)
mask_bool_t = torch.tensor(mask)
tokens_batch_t = torch.tensor(tokens_batch, dtype=torch.long)
targets_batch_t = torch.tensor(targets_batch, dtype=torch.long)

logits_list_t = [mini_gpt_forward_torch(tokens_batch_t[b], params_t, mask_bool_t, NUM_HEADS) for b in range(BATCH_B)]
logits_torch = torch.stack(logits_list_t)
loss_torch = torch.mean(torch.stack([
    cross_entropy_loss_torch(logits_list_t[b], targets_batch_t[b]) for b in range(BATCH_B)
]))

max_logit_diff = np.abs(logits_np - logits_torch.numpy()).max()
loss_diff = abs(loss_np - loss_torch.item())
print(f"max abs diff, logits (NumPy vs. PyTorch): {max_logit_diff:.2e}")
print(f"abs diff, loss       (NumPy vs. PyTorch): {loss_diff:.2e}")
assert max_logit_diff < 1e-9
assert loss_diff < 1e-9

Built from identical weights, the from-scratch NumPy mini-GPT and an
equivalent PyTorch computation agree on every logit and on the loss to
floating-point precision — the same standard 9a's attention and 10a's
Transformer block were held to, now applied to the complete forward pass
and loss of a decoder-only language model. 11b takes this exact
architecture, adds a backward pass (via PyTorch's autograd, not derived by
hand here), and actually trains it.

## Key Takeaways

- **The autoregressive objective is the chain rule of probability**,
  parameterised by one shared conditional $p_\theta(x_t\mid x_{<t})$ per
  position; its negative log-likelihood **is** the classification
  cross-entropy from 2a/8a, applied once per position under 10a's causal
  mask — confirmed to match exactly, not just in the abstract.
- **Perplexity is $\exp(\text{loss})$**, and an untrained (uniform-guessing)
  model's perplexity equals the vocabulary size exactly — a concrete,
  verified calibration point for judging any trained model's perplexity.
- **Loss falls smoothly with parameters and with data**, measured directly
  on a tiny character-level model, the same qualitative shape Kaplan and
  Chinchilla report at vastly larger scale; overall training compute is
  well approximated by $C \approx 6ND$, which is what ties a fixed
  compute budget to a tradeoff between model size and data.
- **A complete decoder-only mini-GPT** — embeddings, sinusoidal positional
  encoding, causal pre-norm Transformer blocks, and an output head,
  assembled entirely from 9a/10a's derived pieces — has its forward pass
  and loss verified against PyTorch to floating-point precision, on a real
  batch of Tiny Shakespeare. 11b trains this exact architecture.